# Project 2: The Analyst Intelligence Terminal — From Messy Filings to Semantic Search

**DSC 785-001: Text Analytics — Summer 2026**
**Project 2A** (retrieval terminal) **+ Project 2B** (RAG Q&A) — one continuous build in one notebook.
**Checkpoint due 6/25** (show your code + a MongoDB screenshot so we can confirm you're on track);
**everything due 6/30.** *(Absorbs the semantic-search material originally planned for Project 3.)*
**Groups of 3–5 students**

**Working copy:** open this notebook in Colab, then File → "Save a copy in Drive" to work in your own.

**Group Members:** *(fill in your names here)*
- Name 1
- Name 2
- Name 3

---

## How this project is graded — read this first

You are being given **working code**. That is intentional. You are analysts, not software engineers,
and this is a one-month course. The code is the *harness*; it is not the deliverable.

**Your grade comes from your presentation and your ability to explain and defend what the code does
and why.** Anyone can press "Run All." Very few people can stand up and answer: *"Where did that
number come from? Who decided this company is a 'CRITICAL' risk? How do you know your search is
actually good and not just plausible?"*

**Frame the whole deliverable as a presentation to the LRCM Investment Committee** — your group is
showcasing the firm's new document-intelligence capability, and the committee will interrupt with hard
questions. You did **not** gather this data or build the extraction pipeline that made the JSONs — but
in that room you are accountable for it anyway, so you must be able to explain where it came from and
how it was made (see **Part A0**).

So every Part below has four kinds of cells:

- **🛠️ GIVEN** — working code. Read it, run it, understand every line. You will be asked about it.
- **✍️ YOUR TURN** — small fill-ins or short written answers in the notebook.
- **✅ CONCEPT CHECK** — quick gut-check questions to make sure you understood the GIVEN code.
- **🏛️ INVESTMENT COMMITTEE QUESTION** — a hard question from a member of the LRCM Investment
  Committee, the board your group is presenting to. They probe provenance, judgment, and verification,
  not code. **These carry the most weight.**
- **🎤 IN YOUR PRESENTATION** — the specific thing your group must show and defend live.

### Grading rubric

| Component | Weight | What we're looking for |
|---|---|---|
| **Live presentation & concept defense** | **45%** | Can your group *explain and defend* the pipeline? Did you answer the Investment Committee's questions with real reasoning, not hand-waving? |
| **Part G bake-off + Part H case study** | **30%** | Did you define a defensible ground truth, pick a winning model with evidence, and produce *investable*, provenance-backed conclusions? |
| **Working notebook (runs end-to-end)** | **15%** | Does it run? Is the MongoDB database populated? Did you submit a read-only URI? |
| **Concept Checks & written answers** | **10%** | Did you fill in the YOUR TURN / CONCEPT CHECK cells thoughtfully? |

*This rubric applies to **each** of Project 2A (Parts A–H) and Project 2B (Parts I–M). Project 2B's
presentation must include a **live RAG demo**.*

> **A note on AI assistants.** With ChatGPT, writing the first version of a cell takes 3 minutes;
> debugging code you don't understand takes 3 days. We will not debug code you cannot explain. The
> feedback will be: *"The student must understand and debug their own code."* Use AI as a partner,
> not a crutch. Every submission needs an **AI_Disclosure** section.

---

## The Scenario

You are a data engineer / junior analyst at **Low Risk Capital Management (LRCM)**, a quantitative
fund run by Prof. Low. The PM just handed you a stack of **LLM-extracted company analysis JSONs** —
structured summaries of 10-K and 10-Q filings for the S&P 500 universe — and asked you to build a
**queryable intelligence terminal.**

The PM wants to type an investment theme like *"AI infrastructure spending"* or *"biosimilar patent
cliff"* and get back a **ranked list of relevant companies, with evidence**. The terminal must
support:
- **exact keyword matches** (when the PM knows the precise term),
- **fuzzy / semantic matches** (when the PM is exploring a thesis), and
- a **combined (hybrid)** mode that gets the best of both.

The data is rich but messy on purpose: numbers are stored as strings, units vary, and several fields
contain LLM-generated prose. That messiness mirrors real LLM pipelines.

**The pipeline:**

| Part | Title | What you build |
|---|---|---|
| A | Data Loading & Schema Exploration | Load the JSON corpus into MongoDB, understand the schema |
| B | Data Cleaning (Regex & Parsing) | Turn messy string metrics into real numbers |
| C | MongoDB Index Deep Dive | Unique, compound, multikey, text, wildcard indexes |
| D | Querying the Terminal | Nested fields, regex + provenance, aggregation, `$text` |
| E | Atlas Search with BM25 | Lucene-powered keyword search |
| F | Vector Search & Hybrid Retrieval (RRF) | Embeddings, `$vectorSearch`, hybrid fusion |
| G | **The Embedding Bake-Off** | Compare 2 embedding models; decide a winner with *your own* test |
| H | **Investment Theme Case Study** | Use the full stack to research two themes — the 2A capstone |
| | **— PROJECT 2B starts here (due 6/30) —** | |
| I | Chunk the corpus into citable passages | Structure-aware chunking for grounded answers |
| J | Build the augmented (RAG) prompt | Hybrid-retrieve context + assemble the prompt |
| K | Generate answers with DeepSeek | Manual paste (free) **or** the ~$5 DeepSeek API |
| L | RAG evaluation | Faithfulness, relevance, coherence — grounded in provenance |
| M | Live Q&A demo | Real questions, including one the corpus *can't* answer |

**Parts A–H = Project 2A; Parts I–M = Project 2B.** **6/25 checkpoint:** submit your code and a MongoDB
screenshot of your populated cluster so we can confirm you're on track (formative — not your final
grade). **Final (everything) due 6/30.** Parts A–F are the required 2A core (heavily scaffolded);
**Parts G, H, and all of 2B (I–M) are where your grade separates** — they are about judgment,
provenance, and defense, not code.

**Submission artifacts:** your notebook **and** your **MongoDB connection URI (read-only user)**.
The database is the primary deliverable. **Standing up your own free MongoDB Atlas cluster is part of
the project** — each group builds and owns its own cluster; do not share one.

## Setup

In [ ]:
!pip install pymongo sentence-transformers -q

import pymongo
from pymongo import errors
import json, os, re
import pandas as pd
import numpy as np
from pprint import pprint

---
# Part A: Data Loading & Schema Exploration

Each JSON is one company: financial metrics, SWOT, risk-factor changes, a bull/bear scorecard,
catalysts, and red flags — all extracted from SEC 10-K/10-Q filings by an LLM pipeline.

### A0. Where this data comes from — and why you must explain it

You did **not** build this dataset; Prof. Low's pipeline did, and the JSONs are handed to you. But in
the boardroom you are accountable for it, so you must be able to explain — at a high level — **how each
JSON was made:**

- An LLM was given the raw **10-K and 10-Q** filings for each company plus a **carefully tuned prompt**
  that told it to role-play an equity analyst and emit a fixed JSON structure: `executive_summary`,
  `key_metrics`, an `mda_deep_dive`, `risk_factor_changes`, a **SWOT** assessment, a
  `bull_bear_scorecard`, catalysts, and **`red_flags`** each with a `severity`.
- The prompt was **iterated many times** (the production version is the *middle-sized* one: too much
  detail made it over-fit historical crises and miss novel ones; too little made it flag everything).
  It was told to treat the filing as the **present moment** and not use post-filing knowledge.
- **`provenance`** records which filings — and dates — each analysis was drawn from (a 10-K plus a more
  recent 10-Q), so a "new risk" means *new since the prior filing.*
- Crucially, the analytical fields (`swot`, `red_flags`, severities) are the **model's judgment, not
  text copied from the filing.** They can be wrong: a **reasoning error** (misreads a sentence) or a
  **hallucination** (a figure that isn't in the filing at all).

> **🎤 IN YOUR PRESENTATION (open with this).** In 2–3 minutes, explain to the committee how these
> JSONs were produced and what their limitations are — *as if you owned the pipeline.* The committee
> will not trust your search results until they trust your data.

### A1. Download and inspect

> **🛠️ GIVEN**

In [ ]:
# Data download removed: the course corpus (text-analytics-jsons.tar.gz) was shared privately with the class
# and is not redistributed here (see README).
!tar -xzf text-analytics-jsons.tar.gz

json_files = [f for f in os.listdir('text-analytics-jsons') if f.endswith('.json')]
print(f'Total company JSONs: {len(json_files)}')
print(f'Sample files: {sorted(json_files)[:10]}')

### A2. Inspect a single document

> **✍️ YOUR TURN — answer BEFORE running the next cell.**
> What fields would you *expect* in a structured analysis of a 10-K? List at least 5.
>
> *(write here)*

> **🛠️ GIVEN**

In [ ]:
doc = json.loads(open('text-analytics-jsons/NVDA.json').read())

print('Top-level keys:')
for key, val in doc.items():
    if isinstance(val, dict):
        print(f'  {key}: dict -> {list(val.keys())[:5]}')
    elif isinstance(val, list):
        print(f'  {key}: list with {len(val)} items')
    else:
        s = str(val)
        print(f'  {key}: {type(val).__name__} -> {s[:70]}')

print('\n--- key_metrics.revenue ---'); pprint(doc['key_metrics']['revenue'])
print('\n--- swot.strengths[:2] ---');  pprint(doc['swot']['strengths'][:2])
print('\n--- risk_factor_changes.new_risks[:2] ---'); pprint(doc['risk_factor_changes']['new_risks'][:2])
pd.DataFrame(doc['provenance']['unit_accessions'])

> **✅ CONCEPT CHECK**
> 1. How many levels of nesting does this document have? Give a deeply nested example.
> 2. What *type* is `key_metrics.revenue.value`? What is it *actually*? Why is that a problem?
> 3. Could you store this in a relational table (Postgres)? What would you lose?

> **🏛️ INVESTMENT COMMITTEE QUESTION — provenance of the data itself.**
> The `swot`, `red_flags`, and `risk_factor_changes` fields were **not copied from the filing** —
> they are an LLM's *interpretation* of it. Open `doc['provenance']` and note that this analysis
> spans a 10-K **and** a 10-Q from specific filing dates. In your presentation, be ready to say:
> *for one specific claim in this JSON, is it a fact stated in the filing, or a judgment the model
> made?* An auditor never accepts a figure without knowing its source. Neither should the PM.

> **🎤 IN YOUR PRESENTATION**
> Walk us through ONE company's document live. Point to a number, a model-generated judgment, and
> the provenance (which filing, which date) behind it.

---
# Part B: Data Cleaning — Regex & Numeric Parsing

The extractor stored every metric as a **string**. Before you can sort, filter, or aggregate, you
must parse them into numbers. The data has comma thousands (`"15,002"`), percentages (`"+12.4%"`),
mixed units (`"USD millions"` vs `"USD billions"` vs non-USD currencies), parenthetical junk in the
unit field, and nulls (`"NOT DISCLOSED"`).

### B1. Survey the mess *before* writing code

> **🛠️ GIVEN** — connect and survey distinct units (this is the "know your data first" step).

In [ ]:
# Build your OWN free MongoDB Atlas cluster — standing it up is part of the project.
# Get this string from Atlas -> Connect -> Drivers. Never commit real credentials to a shared notebook;
# in production you would read it from an environment variable or secrets manager.
uri = 'mongodb+srv://<your_user>:<your_password>@<your-cluster>.mongodb.net/'
client = pymongo.MongoClient(uri)
col = client['TextAnalyticsDB']['10K_Analysis']

# (run A3 first if the collection is empty — see below)
for field in ['revenue', 'net_income', 'free_cash_flow']:
    units = col.distinct(f'key_metrics.{field}.unit')
    print(f'\n{field} ({len(units)} distinct units):')
    for u in sorted(units, key=str)[:12]:
        print(f'  [{col.count_documents({f"key_metrics.{field}.unit": u}):4d}] {u}')

### B2. The parser (GIVEN — read every line, you will be quizzed on it)

> **🛠️ GIVEN**

In [ ]:
def parse_metric_value(value_str, unit_str):
    """USD amounts normalized to MILLIONS. Non-USD -> None. %/per-share returned as-is. None if junk."""
    if not value_str or not isinstance(value_str, str): return None
    if value_str.strip().upper() in ('NOT DISCLOSED','N/A','NA','-',''): return None

    cleaned = value_str.replace(',', '').strip()
    neg = cleaned.startswith('(') and cleaned.endswith(')')      # accounting negative
    if neg: cleaned = cleaned.strip('()')
    m = re.search(r'([+-]?\d+\.?\d*)', cleaned)
    if not m: return None
    num = float(m.group(1)) * (-1 if neg else 1)

    if not unit_str or not isinstance(unit_str, str): return num
    base = re.split(r'\(', unit_str)[0].strip().lower()          # drop parenthetical metadata

    non_usd = ['cad','eur','gbp','jpy','rmb','brl','krw','aud','chf','sek','sgd','hkd','inr','r$']
    if any(c in base for c in non_usd): return None               # design choice: USD-only ranking
    if 'percent' in unit_str.lower() or '/share' in unit_str.lower(): return num

    if   'billion'  in base: num *= 1000
    elif 'thousand' in base: num /= 1000
    elif 'million'  in base: pass
    elif base in ('usd','usd '): num /= 1_000_000
    return num

def parse_percentage(pct_str):
    if not pct_str or not isinstance(pct_str, str): return None
    if pct_str.strip().upper() in ('N/A','NA','-','NOT DISCLOSED',''): return None
    cleaned = pct_str.replace('%','').strip()
    neg = cleaned.startswith('(') and cleaned.endswith(')')
    if neg: cleaned = cleaned.strip('()')
    m = re.search(r'([+-]?\d+\.?\d*)', cleaned)
    return float(m.group(1)) * (-1 if neg else 1) if m else None

> **✍️ YOUR TURN — test before you trust.** Run at least 5 edge cases through each function
> (a billion, an accounting negative `(15,002)`, a non-USD value, a percentage, a `NOT DISCLOSED`)
> and confirm the outputs are what you expect. Paste your test cell.

> **🛠️ GIVEN** — apply the parser and write a clean `metrics_clean` sub-document (preserves originals).

In [ ]:
fields = ['revenue','net_income','eps_diluted','operating_margin','free_cash_flow','total_debt']
for d in col.find({}):
    clean, km = {}, d.get('key_metrics', {})
    for f in fields:
        v = km.get(f, {})
        p = parse_metric_value(v.get('value'), v.get('unit')) if v else None
        if p is not None: clean[f] = p
    if km.get('revenue', {}).get('yoy_change'):
        clean['revenue_yoy_pct'] = parse_percentage(km['revenue']['yoy_change'])
    bbs = d.get('bull_bear_scorecard', {})
    clean['overall_score']  = bbs.get('overall_score')
    clean['overall_signal'] = bbs.get('overall_signal')
    col.update_one({'_id': d['_id']}, {'$set': {'metrics_clean': clean}})

# Verify: this query only works if revenue is a real number now
for d in col.find({'metrics_clean.revenue': {'$gt': 10000}},
                  {'ticker':1,'metrics_clean.revenue':1,'_id':0}
                 ).sort('metrics_clean.revenue', -1).limit(10):
    print(f"  {d['ticker']}: ${d['metrics_clean']['revenue']:,.0f}M")

> **✅ CONCEPT CHECK**
> 1. Why do we normalize everything to **millions** instead of leaving billions as billions?
> 2. We set non-USD values to `None`. What did that design decision *cost* us? Who disappears from the ranking?
> 3. Why store `metrics_clean` as a *new* field instead of overwriting `key_metrics`?

> **🏛️ INVESTMENT COMMITTEE QUESTION — the silent dropouts.**
> Dropping non-USD filers and "NOT DISCLOSED" values makes the data clean — but a clean dataset is
> not a complete one. In an audit, what you *exclude* matters as much as what you keep. In your
> presentation: how many companies did your parser silently drop, and could that bias the PM toward
> (or away from) a whole category of companies (e.g., foreign issuers)?

> **🎤 IN YOUR PRESENTATION**
> Show your parse-error log. Pick one failure and explain whether it was a *fixable bug* or
> *genuinely missing data*, and how you'd handle it in production.

*(A3 — initial load, for reference: create a unique index on `ticker`, then `insert_many(..., ordered=False)`
inside a `try/except errors.BulkWriteError` so re-runs skip duplicates instead of crashing.)*

---
# Part C: MongoDB Index Deep Dive

An index turns an O(n) collection scan (COLLSCAN) into an O(log n) index scan (IXSCAN). The number
to watch in `.explain()` is **`totalDocsExamined`**.

> **🛠️ GIVEN** — create each index type and prove it's used.

In [ ]:
col.create_index([('ticker', 1)], unique=True)                                    # 1. unique
col.create_index([('bull_bear_scorecard.overall_signal', 1),
                  ('metrics_clean.revenue', -1)])                                  # 2. compound
col.create_index([('swot.threats.item', 1)])                                      # 3. multikey (array)
col.create_index([('executive_summary','text'), ('analyst_notes','text')],
                 weights={'executive_summary':5, 'analyst_notes':10},
                 name='text_exec_analyst')                                         # 4. weighted text
col.create_index([('risk_factor_changes.$**', 1)])                                # 5. wildcard

for idx in col.list_indexes():
    print(idx['name'], dict(idx['key']), '| weights:', idx.get('weights'))

# Prove the compound index works:
e = col.find({'bull_bear_scorecard.overall_signal':'STRONG_BULL'}
            ).sort('metrics_clean.revenue', -1).explain()['executionStats']
print('totalDocsExamined:', e['totalDocsExamined'], '| nReturned:', e['nReturned'])

> **✅ CONCEPT CHECK**
> 1. A collection can have only **one** text index but many regular indexes — why?
> 2. The wildcard index is ~50× larger than a single-field index (5 MB vs 102 KB). When is that worth it?
> 3. What does `weights={'executive_summary':5,'analyst_notes':10}` actually change in a search result?

> **🎤 IN YOUR PRESENTATION**
> For each of your 5 indexes, give the one query pattern it optimizes — in one sentence each.

---
# Part D: Querying the Analyst Terminal

### D1. Nested-field queries

> **🛠️ GIVEN**

In [ ]:
for d in col.find({'metrics_clean.revenue_yoy_pct': {'$gt': 0},
                   'metrics_clean.free_cash_flow':  {'$gt': 0}},
                  {'ticker':1,'metrics_clean.revenue_yoy_pct':1,'metrics_clean.free_cash_flow':1,'_id':0}
                 ).sort('metrics_clean.revenue_yoy_pct', -1).limit(10):
    mc = d['metrics_clean']
    print(f"  {d['ticker']}: growth {mc['revenue_yoy_pct']:.1f}%, FCF ${mc['free_cash_flow']:,.0f}M")

### D2. Regex + **provenance highlighting**

> **🛠️ GIVEN** — find "$X billion" mentions and **highlight them in context** (not just extract them).

In [ ]:
import textwrap
pattern = r'\$[\d.]+\s*billion'
for d in col.find({'executive_summary': {'$regex': pattern, '$options':'i'},
                   'metrics_clean.revenue': {'$gt': 10000}},
                  {'ticker':1,'executive_summary':1,'_id':0}).limit(10):
    hl = re.sub(pattern, lambda m: f'\033[1;33m{m.group()}\033[0m', d['executive_summary'], flags=re.I)
    print(f"\n{d['ticker']}:"); print(textwrap.fill(hl, 90))

> **🏛️ INVESTMENT COMMITTEE QUESTION — show me the source.**
> When the terminal returned three numbers for Humana — 37.7, 1.184, and 1 — a committee member asked what
> they *were*. Extracting matches isn't enough; you have to **see them in context**. (They turned out
> to be revenue vs. net income.) This is *provenance*: an LLM can give a reasoning error (misread the
> sentence) or a hallucination (a number not in the text at all). In your presentation, take one
> highlighted match and prove, from the surrounding sentence, what the number actually refers to.

> **🛠️ GIVEN** — critical red flags via `$elemMatch` (exact match on an array element).

In [ ]:
for d in col.find({'red_flags': {'$elemMatch': {'severity':'CRITICAL'}},
                   'metrics_clean.revenue': {'$gt': 20000}},
                  {'ticker':1,'red_flags.flag':1,'red_flags.severity':1,'_id':0}):
    for rf in d['red_flags']:
        if rf.get('severity') == 'CRITICAL':
            print(f"  {d['ticker']}: {rf['flag'][:80]}")

> **🏛️ INVESTMENT COMMITTEE QUESTION — who decided this is "CRITICAL"?**
> `severity: "CRITICAL"` is **not an SEC term** — it's a label the LLM produced from a prompt. Real
> audit concepts (material weakness in internal control, going-concern doubt, a disclaimer of
> opinion, a Critical Audit Matter) *are* defined and *are* required disclosures. In your
> presentation: for one "CRITICAL" company, is the flag backed by a real SEC-defined concept in the
> filing, or is it the model's editorializing? How would you verify?

### D3. Aggregation pipeline — and a real bug worth understanding

> **🛠️ GIVEN** — *count* critical flags per company with `$project` + `$filter` + `$size`.

In [ ]:
pipeline = [
  {'$match': {'red_flags.severity': 'CRITICAL'}},
  {'$project': {'_id':0, 'ticker':1,
     'critical_count': {'$size': {'$filter': {
         'input':'$red_flags', 'as':'f', 'cond':{'$eq':['$$f.severity','CRITICAL']}}}}}}
]
pd.DataFrame(col.aggregate(pipeline)).sort_values('critical_count', ascending=False).head()

> **✅ CONCEPT CHECK — the "all counts = 1" trap.**
> In class, MongoDB's own query-generator produced a pipeline that returned `1` for every company,
> because it descended into the `critical` element and couldn't climb back up to count the parent
> array. The fix (which DeepSeek found) was to **`$unwind`** first. A committee member also flagged a
> possible **"Critical Audit Matter"** confound. In your own words: why does the GIVEN pipeline above
> get the right answer *without* `$unwind`, and how would you *verify* a count is correct rather than
> just plausible?

### D4. `$text` search with relevance score

> **🛠️ GIVEN**

In [ ]:
def text_search(query, limit=10):
    res = list(col.find({'$text': {'$search': query}},
                        {'ticker':1, 'score':{'$meta':'textScore'},'_id':0}
                       ).sort([('score', {'$meta':'textScore'})]).limit(limit))
    print(f'\n$text: "{query}"'); [print(f"  {d['ticker']}: {d['score']:.2f}") for d in res]
    return res

text_search('biosimilar patent competition')
text_search('AI data center infrastructure spending')

> **🎤 IN YOUR PRESENTATION**
> Run 3 searches. For each: are the top hits what you expected? What *relevant* company is missing,
> and *why* might keyword search have missed it? (Foreshadows Parts E–G.)

---
# Part E: Atlas Search with BM25

`$text` is basic term-frequency search. **Atlas Search** runs **BM25** (term frequency × inverse
document frequency, **plus** document-length saturation, fuzzy matching via stemming/lemmatization,
and compound must/should/mustNot queries). This is the production upgrade.

> **Stemming** (chop suffixes): played/plays/playing → play.   **Lemmatization** (dictionary root): are/is → be.

### E1. Create the index (Atlas UI — cannot be done from PyMongo)

1. Cluster → Collections → `10K_Analysis` → **Atlas Search** tab → **Create Search Index** → **JSON Editor**
2. Definition: `{ "mappings": { "dynamic": true } }`  → name it **`default`** → Create → wait for **Active**.

> **🛠️ GIVEN** — free clusters allow only ~3 search indexes, so use them sparingly. Take a screenshot
> of the **Active** index for your submission.

### E2. Run BM25 with `$search`

> **🛠️ GIVEN**

In [ ]:
def atlas_search(query, path='executive_summary', limit=10):
    res = list(col.aggregate([
        {'$search': {'index':'default', 'text': {'query':query, 'path':path}}},
        {'$project': {'ticker':1, 'score':{'$meta':'searchScore'},'_id':0}},
        {'$limit': limit}]))
    print(f'\nBM25: "{query}" on {path}'); [print(f"  {d['ticker']}: {d['score']:.4f}") for d in res]
    return res

text_search('quantum computing')      # old TF-IDF
atlas_search('quantum computing')     # BM25 — compare the two

### E3. Compound query (must / should / mustNot)

> **🛠️ GIVEN** — "growth companies, but not the ones restructuring."

In [ ]:
res = col.aggregate([
  {'$search': {'index':'default', 'compound': {
     'must':    [{'text': {'query':'revenue growth', 'path':'executive_summary'}}],
     'should':  [{'text': {'query':'AI data center', 'path':'analyst_notes'}}],
     'mustNot': [{'text': {'query':'restructuring impairment write-down', 'path':'executive_summary'}}]}}},
  {'$project': {'ticker':1, 'score':{'$meta':'searchScore'},'_id':0}}, {'$limit':10}])
[print(f"  {d['ticker']}: {d['score']:.4f}") for d in res]

> **✅ CONCEPT CHECK** — In one sentence each: what do `must`, `should`, and `mustNot` do to a
> document's *eligibility* vs. its *score*?

> **🎤 IN YOUR PRESENTATION**
> Run the *same* query through `$text` and `$search`. Where do the top-10 lists agree and disagree,
> and **which do you trust more, and why?**

---
# Part F: Vector Search & Hybrid Retrieval (RRF)

BM25 finds documents that contain the *words*. But *"revenue beat expectations"* and *"top-line
exceeded consensus"* mean the same thing with zero shared keywords. **Vector search** embeds text
into a vector that captures *meaning*; similar meanings → nearby vectors. **Hybrid retrieval**
combines BM25 + vector with **Reciprocal Rank Fusion (RRF)**.

### F1. Build an "investable text" and embed it

> **🛠️ GIVEN** — collapse each company's narrative into one string, then embed with the course model.

In [ ]:
import sentence_transformers

def investable_text(doc):
    parts = [doc.get('executive_summary',''), doc.get('analyst_notes','')]
    for sec in ['strengths','weaknesses','opportunities','threats']:
        parts += [it.get('item','') for it in doc.get('swot',{}).get(sec,[])]
    for r in doc.get('risk_factor_changes',{}).get('new_risks',[]):
        parts.append(r.get('risk','') if isinstance(r,dict) else str(r))
    return ' '.join(p for p in parts if p)

minilm = sentence_transformers.SentenceTransformer('all-MiniLM-L6-v2')   # 384-dim, 512-token window

for d in col.find({'embedding_minilm': {'$exists': False}}):            # self-healing: only the to-do's
    vec = minilm.encode(investable_text(d), normalize_embeddings=True).tolist()
    col.update_one({'_id': d['_id']}, {'$set': {'embedding_minilm': vec}})
print('done embedding')

### F2. Create a Vector Search index (Atlas UI), then query it

Atlas Search tab → Create Search Index → **Vector Search** → JSON:
```json
{ "fields": [ { "type": "vector", "path": "embedding_minilm", "numDimensions": 384, "similarity": "cosine" } ] }
```
Name it **`vector_index`**, wait for **Active**.

> **🛠️ GIVEN**

In [ ]:
def vector_search(query, limit=10):
    qv = minilm.encode(query, normalize_embeddings=True).tolist()
    res = list(col.aggregate([
        {'$vectorSearch': {'index':'vector_index','path':'embedding_minilm',
                           'queryVector':qv,'numCandidates':100,'limit':limit}},
        {'$project': {'ticker':1,'score':{'$meta':'vectorSearchScore'},'_id':0}}]))
    print(f'\nVECTOR: "{query}"'); [print(f"  {d['ticker']}: {d['score']:.4f}") for d in res]
    return res

vector_search('companies building physical AI data center capacity')

### F3. Hybrid retrieval with RRF (done in Python — free Mongo has no native hybrid)

> **🛠️ GIVEN** — RRF just averages **ranks**, not scores (you can't add a BM25 score to a cosine score,
> the way you can't add a P/E ratio to an ROE).

In [ ]:
def rrf(*rank_lists, k=60, limit=10):
    scores = {}
    for ranks in rank_lists:                       # each ranks = ['AAPL','MSFT', ...] best-first
        for pos, ticker in enumerate(ranks):
            scores[ticker] = scores.get(ticker, 0) + 1.0/(k + pos + 1)
    return sorted(scores, key=scores.get, reverse=True)[:limit]

q = 'AI data center infrastructure spending'
bm25_ranks   = [d['ticker'] for d in atlas_search(q, limit=20)]
vector_ranks = [d['ticker'] for d in vector_search(q, limit=20)]
print('\nHYBRID (RRF):', rrf(bm25_ranks, vector_ranks))

> **✅ CONCEPT CHECK**
> 1. Give a query where vector search should beat BM25, and one where BM25 should beat vector. Why?
> 2. Why does RRF combine **ranks** instead of raw scores?

> **🎤 IN YOUR PRESENTATION**
> Show one theme where **hybrid** beats *both* BM25 and vector alone, and explain what each method
> contributed.

---
# Part G: The Embedding Bake-Off 🏆

> *"At LRCM we don't operate on feelings, we operate on data. You picked `all-MiniLM` because it's
> the course default. Prove it's the right choice — or replace it."* — Prof. Low

You will compare **two** embedding models and **decide a winner with a test you design and defend.**

### G1. The two contenders

| | Model | Dim | Context window | Note |
|---|---|---|---|---|
| **Generalist** | `all-MiniLM-L6-v2` | 384 | ~512 tokens (~350 words) | fast, course default; silently truncates long text |
| **Specialist** | `nomic-ai/nomic-embed-text-v1.5` | 768 | ~8192 tokens | needs a **task prefix** (e.g. `search_document:`) |

> **🛠️ GIVEN** — generate the second model's embeddings (nomic), stored alongside the first.

In [ ]:
nomic = sentence_transformers.SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)

for d in col.find({'embedding_nomic': {'$exists': False}}):
    text = 'search_document: ' + investable_text(d)          # nomic REQUIRES a task prefix
    vec = nomic.encode(text, normalize_embeddings=True).tolist()
    col.update_one({'_id': d['_id']}, {'$set': {'embedding_nomic': vec}})
print('done')

### G2. 🏛️ INVESTMENT COMMITTEE QUESTION (this is the whole point of Part G)

> *"How do you KNOW one model is better? Where is the ground truth?"*
>
> There is no answer key shipped with this data. **So you will build one.** Pick **3–5 investment
> themes** and, for each, hand-pick a small **truth set** of tickers that genuinely belong — and be
> ready to **defend why each one is in or out.** This is exactly the judgment an analyst is paid for.
> Example (you must justify yours):
>
> ```python
> THEMES = {
>   'quantum computing':        {'IONQ', 'RGTI', 'QBTS', 'QUBT'},     # pure-plays; defend exclusions
>   'biosimilar patent cliff':  {'REGN', 'AMGN', 'ORGN'},            # who counts? who's borderline?
>   'AI data center power':     {'VRT', 'ETN', 'CEG', 'NRG'},
> }
> ```
> **Document your inclusion rule.** Is HPE a "quantum computing" company because it has a tiny quantum
> unit? Your truth set encodes a real analytical opinion — own it.

### G3. Score each model with **Retrieval@K** (your test)

> **🛠️ GIVEN** — pure cosine similarity in numpy (no extra Atlas index needed for the bake-off).

In [ ]:
def load_matrix(field):
    rows = list(col.find({field: {'$exists': True}}, {'ticker':1, field:1, '_id':0}))
    tickers = [r['ticker'] for r in rows]
    M = np.array([r[field] for r in rows])                 # already normalized -> cosine = dot product
    return tickers, M

def retrieval_at_k(model_obj, field, query_text, truth, k=10, nomic=False):
    tickers, M = load_matrix(field)
    q = ('search_query: ' + query_text) if nomic else query_text
    qv = model_obj.encode(q, normalize_embeddings=True)
    top = [tickers[i] for i in np.argsort(-(M @ qv))[:k]]  # top-k by cosine
    hits = truth & set(top)
    return len(hits) / len(truth), top, hits               # recall of the truth set within top-k

scoreboard = []
for theme, truth in THEMES.items():
    s_mini, top_mini, _ = retrieval_at_k(minilm, 'embedding_minilm', theme, truth, k=10)
    s_nom,  top_nom,  _ = retrieval_at_k(nomic,  'embedding_nomic',  theme, truth, k=10, nomic=True)
    scoreboard.append({'theme':theme, 'MiniLM@10':s_mini, 'Nomic@10':s_nom})
pd.DataFrame(scoreboard)

> **✅ CONCEPT CHECK**
> 1. `all-MiniLM` truncates at ~350 words but your `investable_text` is longer — what gets *silently
>    dropped*, and could that help or hurt on these themes?
> 2. Why must you prefix nomic with `search_query:` for the query but `search_document:` for the stored
>    text? What happens if you mix them up?

### G4. Now design your own experiments & evaluation standard (required)

The Retrieval@K test above is the one we built **in class** — it is *one* opinion about "better." A real
quant team never settles a question on a single metric. **Design and run 2–3 *additional* experiments
of your own**, changing **one knob at a time**, and write down **your own evaluation standard** for
declaring a winner. The goal is a recommendation for the **best end-to-end DB configuration** —
*which model, which text, which retrieval mode* — for LRCM's terminal.

High-level directions (pick 2–3, or invent your own):

- **Change the input text.** `executive_summary` only vs. your full `investable_text` vs. `analyst_notes`
  only. Does more text help, or dilute the signal (recall MiniLM's ~350-word truncation)?
- **Change K.** Does the winner hold at Retrieval@5, @10, @25? A model that only wins at large K may
  just be getting lucky.
- **Compare retrieval *modes*, not just models.** Score `$text` vs. BM25 vs. vector vs. hybrid (RRF) on
  your themes. Sometimes the *method* matters more than the model.
- **Test the nomic prefixes.** `search_document:`/`search_query:` vs. `classification:`/`clustering:` —
  does the prefix shift the ranking on your themes?
- **Borrow an independent ground truth.** Join Project 1's `ivv_holdings_clean.csv` `Sector` column by
  ticker for a **GICS-sector silhouette**, or build a **market-correlation precision@k** from
  `yfinance` returns. Does an outside yardstick agree with your hand-built theme test? If not, which do
  you believe, and why?

> **✍️ YOUR TURN — write your evaluation standard FIRST, before running anything.** One paragraph:
> *what will make you declare a configuration the winner?* (e.g., "highest mean Retrieval@10 across our
> 4 themes, provided it also wins on at least one independent metric and doesn't collapse at K=5.")
> Then run your experiments and hold yourself to it.

> **🏛️ INVESTMENT COMMITTEE QUESTION.** A metric you chose *after* seeing the results is not evidence — it is a
> rationalization (the auditor's version of cherry-picking). Be ready to show the committee that you set
> your standard **before** you ran the experiments, and that you would have accepted the answer even if
> it went against the course-default model.

> **🎤 IN YOUR PRESENTATION — the bake-off verdict**
> 1. Show your scoreboards — the class Retrieval@K test **and** your 2–3 own experiments.
> 2. **Defend your truth sets and your written evaluation standard.** Which ticker was the hardest call?
>    Which experiment surprised you?
> 3. Recommend the **end-to-end DB configuration** LRCM should standardize on — model, input text, and
>    retrieval mode — justified by your numbers, not vibes.

---
# Part H: Investment Theme Case Study (Capstone) 🎯

This is the deliverable the PM actually cares about. Using your **chosen** stack (BM25 + vector +
hybrid, with the winning embedding model from Part G), research **two** themes from LRCM's mandate —
**one your group finds compelling, one you're skeptical of.**

LRCM thematic mandates (pick two): *AI infrastructure & chips · cloud / data centers / power ·
nuclear & grid storage · crypto & blockchain · quantum & advanced semis · robotics & autonomy ·
AI cybersecurity · fintech & payments · gene editing & AI drug discovery.*

For **each** theme, your group must deliver:

1. **The search.** Run the theme through `$text`, BM25, vector, and hybrid. Show how the ranked lists
   differ. State which retrieval method you trust for *this* theme and why.
2. **The shortlist.** Pick 3–5 companies. For each, give the **evidence** — quote the specific
   `executive_summary` / `swot` / `red_flags` text, and **highlight where in the source it came from.**
3. **The provenance & verification.** For your single strongest claim, trace it to the filing
   (provenance: 10-K vs 10-Q, date). Flag any claim that is the *model's* judgment rather than a
   stated fact.
4. **The red-flag screen.** Run your shortlist through the CRITICAL red-flag screen (Part D). Did any
   "great theme" name carry a material-weakness / going-concern / disclaimer flag? What would you tell
   the PM?
5. **The recommendation.** One slide: would LRCM build a position, pass, or short — and the one thing
   that would change your mind.

> **🏛️ INVESTMENT COMMITTEE QUESTION — the final exam.**
> *"If the PM acts on this and loses money, can you trace every claim back to a filing?"* Your case
> study must survive that question. An analyst who can't show provenance is
> the one who doesn't know who the mark is at the poker table.

> **🎤 IN YOUR PRESENTATION**
> Each group member presents at least one Part live and must be able to field the Investment
> Committee's questions about *any* part — this is not divide-and-conquer. Everyone owns the whole pipeline.

*(End of Project 2A.)*

---
# PROJECT 2B: Intelligent Q&A with Retrieval-Augmented Generation (RAG) — Due 6/30

> *"The terminal in 2A finds the right companies. Now make it answer questions in plain English — and,
> more importantly, make it answer ONLY from what's in our database, with a citation for every claim.
> An analyst who makes things up is worse than no analyst."* — Prof. Low

2B builds **directly on your 2A database, embeddings, and hybrid retrieval** — you do not re-ingest
anything. **RAG = Retrieve** (your 2A hybrid search) **+ Augment** (stuff the retrieved evidence into a
prompt) **+ Generate** (an LLM writes the answer). The whole game is **grounding**: every sentence of
the answer must be traceable to a retrieved passage.

**LLM choice — pick one (we default to DeepSeek, the model that out-reasoned MongoDB's query generator in class):**
- **Free / no code:** print the augmented prompt, paste it into **chat.deepseek.com**, paste the answer
  back into the notebook. Zero dollars, fully manual.
- **~$5 API:** create a DeepSeek API key and let the notebook call it. Same result, automated — and
  required if you want the optional LLM-as-judge evaluation in Part L.
- **Local SLM (free, offline):** run a small model with **Ollama** (e.g. `gemma3n`) on your own
  machine. Fully private, but a small model may follow the "answer only from the context" rule less
  reliably than DeepSeek — which is itself a worthwhile thing to test and report.

## Part I: Chunk the corpus into citable passages

Your 2A retrieval returns whole companies. For a *grounded* answer you need smaller, **citable
passages** so the LLM — and your grader — can point to the exact source. Good news: the extraction
pipeline already chunked the data for you. Every `swot` item, `new_risk`, and `red_flag` is a
self-contained passage. This is **structure-aware chunking**, and it beats splitting on a word count.

> **🛠️ GIVEN**

In [ ]:
def company_passages(doc):
    """Flatten one company JSON into citable passages: (ticker, field, text)."""
    t = doc['ticker']; out = []
    if doc.get('executive_summary'): out.append((t,'executive_summary', doc['executive_summary']))
    if doc.get('analyst_notes'):     out.append((t,'analyst_notes',     doc['analyst_notes']))
    for s in doc.get('swot',{}).get('strengths',[]): out.append((t,'swot.strength', s.get('item','')))
    for s in doc.get('swot',{}).get('threats',[]):   out.append((t,'swot.threat',   s.get('item','')))
    for r in doc.get('risk_factor_changes',{}).get('new_risks',[]):
        out.append((t,'new_risk', r.get('risk','') if isinstance(r,dict) else str(r)))
    for rf in doc.get('red_flags',[]):
        out.append((t, f"red_flag:{rf.get('severity','')}", rf.get('flag','')))
    return [(tk,f,txt) for (tk,f,txt) in out if txt]

> **✅ CONCEPT CHECK** — Why is splitting this JSON by *field* better than splitting the raw text every
> 1,000 words? (What happens to a single SWOT threat if a word-count chunker cuts it in half?)

## Part J: Build the augmented (RAG) prompt — using your 2A hybrid search

> **🛠️ GIVEN** — retrieve with the 2A **hybrid** stack (BM25 + vector + RRF), then assemble the prompt.

In [ ]:
def hybrid_retrieve(query, k=5):
    bm25_ranks   = [d['ticker'] for d in atlas_search(query, limit=20)]    # Part E
    vector_ranks = [d['ticker'] for d in vector_search(query, limit=20)]   # Part F
    return rrf(bm25_ranks, vector_ranks, limit=k)                           # RRF from Part F

def build_rag_prompt(question, k=5):
    tickers = hybrid_retrieve(question, k=k)
    docs = {d['ticker']: d for d in col.find({'ticker': {'$in': tickers}})}
    passages = [p for t in tickers for p in company_passages(docs[t])]
    context = '\n'.join(f"[{i+1}] ({tk} | {f}) {txt}" for i,(tk,f,txt) in enumerate(passages))
    prompt = f"""You are an equity research assistant for Low Risk Capital Management.
Answer the QUESTION using ONLY the numbered CONTEXT passages below.
- Cite the passage number for every claim, like [3].
- If the answer is not in the context, reply EXACTLY: "Not in the provided sources."
- Do not use any outside knowledge.

QUESTION: {question}

CONTEXT:
{context}
"""
    return prompt, passages

prompt, passages = build_rag_prompt('Which companies face new risks from chip export restrictions to China?')
print(prompt)

> **✍️ YOUR TURN** — write 5 questions your PM might actually ask. **At least one must be answerable
> only from outside the corpus** (you'll use it in Part M to test for hallucination).

> **🏛️ INVESTMENT COMMITTEE QUESTION — the line that does the real work.**
> *"If the answer is not in the context, reply 'Not in the provided sources.'"* That single instruction
> is your main defense against hallucination. Be ready to explain to the committee why a RAG system
> with that rule is more trustworthy to an auditor than asking ChatGPT directly.

## Part K: Generate the answer (three paths — pick one)

> **🛠️ GIVEN — Path A: Free, manual (no API key).**

In [ ]:
prompt, passages = build_rag_prompt('your question here')
print(prompt)        # copy this ENTIRE prompt -> paste into chat.deepseek.com -> paste the answer below

answer = """
<paste DeepSeek's answer here, including its [n] citations>
"""
print(answer)

> **🛠️ GIVEN — Path B: ~$5 API (OpenAI-compatible).**

In [ ]:
# pip install openai   ;   get a key at platform.deepseek.com
from openai import OpenAI
client = OpenAI(api_key='YOUR_DEEPSEEK_KEY', base_url='https://api.deepseek.com')

def ask_deepseek(prompt, model='deepseek-chat', temperature=0.2):
    r = client.chat.completions.create(model=model, temperature=temperature,
                                       messages=[{'role':'user','content':prompt}])
    return r.choices[0].message.content

prompt, passages = build_rag_prompt('your question here')
answer = ask_deepseek(prompt)
print(answer)

> **🛠️ GIVEN — Path C: Local SLM via Ollama (free, offline).**

In [ ]:
# install Ollama, then:  ollama pull gemma3n:e4b   ;   pip install ollama
import ollama
def ask_ollama(prompt, model='gemma3n:e4b'):
    r = ollama.chat(model=model, messages=[{'role':'user','content':prompt}],
                    options={'temperature':0.2})
    return r['message']['content']

prompt, passages = build_rag_prompt('your question here')
answer = ask_ollama(prompt)
print(answer)

> **✅ CONCEPT CHECK** — We set `temperature=0.2`, not `0.9`. Why does a research assistant that must
> stick to its sources want *low* temperature? And if you used the local Ollama SLM, did it obey the
> "Not in the provided sources" rule as reliably as DeepSeek? (Report what you saw.)

## Part L: RAG evaluation — faithfulness, relevance, coherence

A fluent answer is worthless if it isn't grounded. Score every answer on three axes:

| Metric | The question it asks | How to check |
|---|---|---|
| **Faithfulness** | Is every claim supported by a cited passage? | For each sentence, find its `[n]` and confirm the passage actually says it. **Any unsupported sentence is a hallucination.** |
| **Answer relevance** | Does it actually answer the question? | Re-read the question — did the answer address it, or dodge? |
| **Coherence** | Is it clear and well organized? | Would you hand it to the PM as-is? |

> **✍️ YOUR TURN — manual faithfulness audit (required).** For one answer, build a table: each
> sentence → its citation `[n]` → the passage text → ✅/❌ "supported." Then score 1–5 on each axis.

> **🛠️ GIVEN (optional, API path) — LLM-as-judge for faithfulness.**

In [ ]:
def judge_faithfulness(answer, prompt):
    judge = f"""You are a strict auditor. For EACH sentence in the ANSWER, decide whether it is FULLY
supported by the CONTEXT inside the PROMPT. List every unsupported sentence as a FAITHFULNESS
VIOLATION, then give an overall faithfulness score from 1 to 5.

PROMPT (with context):
{prompt}

ANSWER:
{answer}"""
    return ask_deepseek(judge)

print(judge_faithfulness(answer, prompt))

> **🏛️ INVESTMENT COMMITTEE QUESTION — faithfulness IS provenance.**
> This is the exact standard from 2A, now applied to a *generated* answer: can you trace every sentence
> back to a source passage? In your demo, take one answer and, live, point to the passage behind each
> claim. The sentence you can't ground is the one that gets the fund sued.

## Part M: Live Q&A demo (the deliverable)

Run your system end-to-end on 4–5 questions. You **must** include:

1. **A normal question** the corpus answers well — show the cited answer *and* the passages behind it.
2. **A comparison question** (e.g., *"Compare the new risks facing NVDA vs. AMD"*) that forces
   multi-company retrieval.
3. **An adversarial / out-of-corpus question** (e.g., *"What is Apple's CEO's home address?"*) — the
   system must reply *"Not in the provided sources,"* **not** invent an answer. If it hallucinates,
   that's a finding: show how you'd fix the prompt or retrieval.

> **🎤 IN YOUR PRESENTATION (boardroom)** — Demo it live. Prove faithfulness on the spot for one answer
> by pointing to each source passage. State honestly where the system is and isn't trustworthy — the
> Investment Committee will probe exactly that.

*(End of Project 2B — due 6/30.)*

---

## Appendix: Prompt Engineering Best Practices (reference)

The `swot`, `red_flags`, and analysis fields in this corpus were produced by an LLM prompt. If you
build your own extraction (or use AI to help anywhere in this project), follow these:

1. **Role / persona** — *"You are an equity analyst extracting material information…"*
2. **Explicit task & goal** — not *"summarize,"* but *"extract the 1–5 most material, stock-relevant points."*
3. **Delimiters** — fence the input with `---` or `"""` to separate instructions from data.
4. **Constraints** — say what you want *and* what you don't (no preamble, no markdown, JSON only).
5. **Few-shot examples** — show 2–3 input→output pairs; the single most effective reliability trick.
6. **Force structure** — request a specific JSON schema (Pydantic enforces it).
7. **Error handling** — *"If no material info is present, return an empty list"* — prevents hallucination.

## Appendix: Submission Checklist

**Checkpoint (due 6/25)**
- [ ] Your notebook code so far (Parts A–F running) **+ a screenshot of your populated MongoDB cluster**, so we can confirm you're on track. *(Formative on-track check — not the final grade.)*

**Project 2A (part of the 6/30 final)**
- [ ] Boardroom opener (Part A0): can explain how the JSONs were produced and their limitations.
- [ ] Your **own** MongoDB Atlas cluster, populated (`metrics_clean`, both embeddings, all indexes); notebook runs end-to-end.
- [ ] **Read-only MongoDB URI** submitted.
- [ ] Atlas Search **and** Vector Search index screenshots (Active).
- [ ] Part G: class Retrieval@K test **+ your 2–3 own experiments**, defended truth sets, and your **written evaluation standard** (set before running).
- [ ] Part H case study: two themes, evidence with provenance, red-flag screen, recommendation.

**Project 2B (due 6/30)**
- [ ] RAG pipeline runs: hybrid retrieval → augmented prompt → DeepSeek answer **with citations**.
- [ ] Manual faithfulness audit of ≥1 answer (sentence → citation → supported?).
- [ ] Live demo covers a normal question, a multi-company comparison, **and** an out-of-corpus question the system refuses to answer.
- [ ] Note which LLM path you used (DeepSeek manual paste, DeepSeek API, or local Ollama SLM).

**Both projects**
- [ ] **AI_Disclosure** section: what AI tools you used, for what, and how you verified the output.
- [ ] Trello board with task assignments.
- [ ] Every member can field an Investment Committee question about any Part.